# Native Raw Interfacing

You can specifically tell fortran to compile with C types so that it and  
python can communicate back and forth with them. 

Although it's not as simple as utilizing f2py later.

Take the following fortran code and save it into `my_fortran_code.f90`

```fortran
module fortran_dll
use iso_c_binding
implicit none

contains

subroutine add_numbers(x, y, result) bind(c)
   real(c_double), value :: x, y  ! `value` makes the arguments passed by value
   real(c_double) :: result       ! This will store the result
   result = x + y
end subroutine add_numbers

subroutine multiply_numbers(x, y, result) bind(c)
   real(c_double), value :: x, y
   real(c_double) :: result
   result = x * y
end subroutine multiply_numbers
end module fortran_dll
```

from the same directory as the fortran code, compile the code into a dll,

```powershell
gfortran -shared -o C:\Users\dane.parks\PycharmProjects\civilpy\src\civilpy\general\fortran\fortran_DLLs\my_fortran_lib.dll my_fortran_code.f90
```

You can access the DLL from python as follows,

In [6]:
!gfortran --version

'gfortran' is not recognized as an internal or external command,
operable program or batch file.


In [5]:
import ctypes

fortran_lib = ctypes.CDLL(r"C:\Users\dane.parks\PycharmProjects\civilpy\src\civilpy\general\fortran\fortran_DLLs\my_fortran_lib.dll")

fortran_lib.add_numbers.argtypes = [ctypes.c_double, ctypes.c_double, ctypes.POINTER(ctypes.c_double)]
fortran_lib.add_numbers.restype = None

fortran_lib.multiply_numbers.argtypes = [ctypes.c_double, ctypes.c_double, ctypes.POINTER(ctypes.c_double)]
fortran_lib.multiply_numbers.restype = None

x = 5.0
y = 3.0
result = ctypes.c_double()  # Create a container object for the result

fortran_lib.add_numbers(ctypes.c_double(x), ctypes.c_double(y), ctypes.byref(result))
print(f"Addition result: {result.value}")

fortran_lib.multiply_numbers(ctypes.c_double(x), ctypes.c_double(y), ctypes.byref(result))
print(f"Multiplication result: {result.value}")

FileNotFoundError: Could not find module 'C:\Users\dane.parks\PycharmProjects\civilpy\src\civilpy\general\fortran\fortran_DLLs\my_fortran_lib.dll' (or one of its dependencies). Try using the full path with constructor syntax.

# Utilizing the `f2py` Python Library

save as `my_fortran_code.f90`

In [4]:
!python -m numpy.f2py -c my_fortran_code.f90 -m my_fortran_lib

Cannot use distutils backend with Python>=3.12, using meson backend instead.
Using meson backend
Will pass --lower to f2py
See https://numpy.org/doc/stable/f2py/buildtools/meson.html
Reading fortran codes...
	Reading file 'my_fortran_code.f90' (format:free)
Post-processing...
	Block: my_fortran_lib
			Block: fortran_dll
				Block: add_numbers
				Block: multiply_numbers
Applying post-processing hooks...
  character_backward_compatibility_hook
Post-processing (stage 2)...
	Block: my_fortran_lib
		Block: unknown_interface
			Block: fortran_dll
				Block: add_numbers
				Block: multiply_numbers
Building modules...
    Building module "my_fortran_lib"...
		Constructing F90 module support for "fortran_dll"...
            Constructing wrapper function "fortran_dll.add_numbers"...
              result = add_numbers(x,y)
            Constructing wrapper function "fortran_dll.multiply_numbers"...
              result = multiply_numbers(x,y)
    Wrote C/API module "my_fortran_lib" to file ".\my_

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\dparks1\PycharmProjects\civilpy\.venv\Lib\site-packages\numpy\f2py\__main__.py", line 5, in <module>
    main()
    ~~~~^^
  File "C:\Users\dparks1\PycharmProjects\civilpy\.venv\Lib\site-packages\numpy\f2py\f2py2e.py", line 781, in main
    run_compile()
    ~~~~~~~~~~~^^
  File "C:\Users\dparks1\PycharmProjects\civilpy\.venv\Lib\site-packages\numpy\f2py\f2py2e.py", line 753, in run_compile
    builder.compile()
    ~~~~~~~~~~~~~~~^^
  File "C:\Users\dparks1\PycharmProjects\civilpy\.venv\Lib\site-packages\numpy\f2py\_backends\_meson.py", line 192, in compile
    self.run_meson(self.build_dir)
    ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^
  File "C:\Users\dparks1\PycharmProjects\civilpy\.venv\Lib\site-packages\numpy\f2py\_backends\_meson.py", line 185, in run_meson
    self._run_subprocess_command(setup_command, build_dir)
    ~~~~~~~~~~~~~~~~